## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-20-19-53-40 +0000,nyt,Netanyahu Associates Acknowledge Role in Setti...,https://www.nytimes.com/2026/11/19/world/middl...
1,2026-11-20-19-51-52 +0000,bbc,Trump threatens ban on US firms building AI da...,https://www.bbc.co.uk/news/articles/cmde1lw9y5...
2,2026-11-20-19-50-02 +0000,nyt,Here’s the latest.,https://www.nytimes.com/live/2026/11/20/us/tru...
3,2026-11-20-19-50-02 +0000,nyt,Britain and Brussels Have Begun a Brexit Rollb...,https://www.nytimes.com/2026/11/20/world/europ...
4,2026-11-20-19-48-40 +0000,nypost,Anthropic’s new IPO millionaires awkward about...,https://nypost.com/2026/11/20/business/anthrop...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2776/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
9,trump,46
28,anthropic,27
29,new,20
14,ai,18
94,tech,18


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
57,2026-11-20-18-06-47 +0000,chitri,Donald Trump says he plans to penalize US tech...,https://www.chicagotribune.com/2026/11/20/trum...,131
147,2026-11-20-14-42-21 +0000,nypost,Trump speaking to press after historic Xi-Puti...,https://nypost.com/2026/11/20/us-news/trump-to...,108
154,2026-11-20-14-28-00 +0000,wsj,The Trump administration auctioned off parts o...,https://www.wsj.com/politics/the-u-s-is-auctio...,106
35,2026-11-20-18-51-22 +0000,nyt,Trump Threatens to Bar U.S. Tech Firms From Bu...,https://www.nytimes.com/2026/11/20/us/politics...,100
31,2026-11-20-19-02-11 +0000,nypost,Anthropic co-founder to make ‘large gift’ to T...,https://nypost.com/2026/11/20/business/anthrop...,98


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
57,131,2026-11-20-18-06-47 +0000,chitri,Donald Trump says he plans to penalize US tech...,https://www.chicagotribune.com/2026/11/20/trum...
4,76,2026-11-20-19-48-40 +0000,nypost,Anthropic’s new IPO millionaires awkward about...,https://nypost.com/2026/11/20/business/anthrop...
271,72,2026-11-20-04-12-00 +0000,wsj,The Senate passed a bipartisan bill designed t...,https://www.wsj.com/politics/policy/senate-pas...
307,70,2026-11-20-00-43-45 +0000,nypost,California state bar opens probe into ex-White...,https://nypost.com/2026/11/19/us-news/californ...
14,64,2026-11-20-19-39-00 +0000,wsj,"Sen. Tom Cotton (R., Ark.), a China hawk, tell...",https://www.wsj.com/politics/national-security...
98,58,2026-11-20-16-41-52 +0000,bbc,Faisal Islam: This cost-cutting UK-EU deal doe...,https://www.bbc.co.uk/news/articles/cm1wr029ev...
147,58,2026-11-20-14-42-21 +0000,nypost,Trump speaking to press after historic Xi-Puti...,https://nypost.com/2026/11/20/us-news/trump-to...
188,46,2026-11-20-12-00-00 +0000,wsj,Vice President JD Vance has whipsawed between ...,https://www.wsj.com/politics/trump-says-vance-...
296,44,2026-11-20-01-37-20 +0000,nyt,Myanmar Committed Genocide Against the Rohingy...,https://www.nytimes.com/2026/11/19/world/europ...
277,39,2026-11-20-03-25-06 +0000,nyt,"Fiona the Hippo, Star of Cincinnati Zoo and Be...",https://www.nytimes.com/2026/11/19/us/fiona-hi...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
